# Probability of LWS
In this notebook, we analyze the probability that a _target visits_ is a **Looking without Seeing** instance, i.e. we calculate $P[\text{LWS} | \text{on target}]$, across different stimuli characteristics:
- **Target Category** is the category of the target object (`animal body`/`animal face`/`human body`/`human face`/`inanimate handmade`/`inanimate natural`)
- **Trial Category** is the presentation mode of the search array (`color`/`grayscale`/`noisy background`)
- **target Rotation** is the (absolute) rotation angle of the target (`0°`,`1°`, ..., `20°`)



In [1]:
import time
import warnings
from itertools import product

import numpy as np
import pandas as pd
import polars as pl
import bambi as bmb
import arviz as az
from pymer4.models import glmer
from scipy.special import expit

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.funnels.build_funnels import build_event_classification_funnel
from analysis.helpers.funnels.size_and_proportion import calculate_step_sizes, calculate_proportions

from analysis.helpers.visualizations.funnel.step_size import step_sizes_figure
from analysis.helpers.visualizations.funnel.category_comparison import category_comparison_figure

pio.renderers.default = "browser"      # or "browser"

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


#### Set Constants

In [2]:
SEED = 42
VERBOSE = True
EVENT_TYPE = "visit"

## Prepare Data
### (1) Read the Data and Apply Funnel Criteria
The `build_event_classification_funnel()` function does the following steps:
1) Read data
2) Exclude bad trials
3) Applies the LWS funnel to remaining visits/fixations

In [3]:
funnel_results = build_event_classification_funnel(
    cnfg.OUTPUT_PATH, "lws", EVENT_TYPE, exclude="invalid_trials"
)

### (2) Sum Criteria Passes
The `calculate_funnel_step_sizes()` function calculates the number of events that pass each criterion in the funnel

In [4]:
funnel_sizes = calculate_step_sizes(
    funnel_results,
    grouping_cols=[
        "subject", "trial", "target", "trial_category", "target_category", "target_angle"
    ],
    criteria_cols=[
        "on_target", "before_identification", "not_close_to_trial_end", "not_before_exemplar_visit", "is_lws"
    ]
)
funnel_sizes = (
    funnel_sizes
    .reset_index(drop=False)
    .assign(abs_target_angle=lambda df: df["target_angle"].abs())
)

## Descriptive Statistics
We calculate the probability of _LWS_ given that the event is _on target_, i.e. $P[\text{LWS} | \text{on target}]$, for the average subject, and the average subject across different trial categories and target categories.

In [5]:
def calc_rates(groups):
    on_target_events = funnel_results.loc[funnel_results["on_target"]]
    return on_target_events.groupby(groups, observed=True)["is_lws"].mean()


subject_lws_rates = calc_rates(["subject"]).agg(["count", "mean", "sem"])
print(f"Average P[LWS | on-target] across subjects = {100 * subject_lws_rates['mean'].mean():.2f}% ± {100 * subject_lws_rates['sem'].mean():.2f}%")

print(f"Average P[LWS | on-target] across Trial Types:")
trialtype_lws_rates = (
    calc_rates(["subject", "trial_category"])
    .groupby("trial_category", observed=True)
    .agg(["mean", "sem"])
)
display(trialtype_lws_rates)

print("Average P[LWS | on-target] across Target Categories:")
targettype_lws_rates = (
    calc_rates(["subject", "target_category"])
    .groupby("target_category", observed=True)
    .agg(["mean", "sem"])
)
display(targettype_lws_rates)

Average P[LWS | on-target] across subjects = 26.24% ± 1.96%
Average P[LWS | on-target] across Trial Types:


,mean,sem
trial_category,,
COLOR,0.232402,0.028127
BW,0.304162,0.024054
NOISE,0.248736,0.019045


Average P[LWS | on-target] across Target Categories:


,mean,sem
target_category,,
ANIMAL_OTHER,0.319524,0.020964
HUMAN_OTHER,0.312531,0.037788
ANIMAL_FACE,0.355464,0.048347
HUMAN_FACE,0.238620,0.022227
OBJECT_HANDMADE,0.190197,0.023283
OBJECT_NATURAL,0.192516,0.025689


## Descriptive Figures
### (1) Visualize Funnel
We visualize how each step in the funnel depletes the initial pool of events until only the subset of events that pass all criteria are left.

In [6]:
funnel_fig = step_sizes_figure(
    funnel_sizes, "on_target", "is_lws",
    title=f"LWS Funnel: {EVENT_TYPE.capitalize()}s",
    show_individuals=False
)

funnel_fig.show()

### (2) P[LWS | on-target] across *Target Types*

In [7]:
target_category_props = calculate_proportions(
    funnel_sizes,
    numerator_col="is_lws", denominator_col="on_target",
    primary_key="subject", subgroups="target_category"
)

target_category_fig = category_comparison_figure(
    target_category_props,
    categ_col="target_category",
    show_distributions=True,
    show_individuals=True,
    show_mean=True,
)

target_category_fig.update_xaxes(
    row=1, col=1,
    tickmode="array",
    tickvals=target_category_props["target_category"].unique(),
    ticktext=[cat.replace("_", "<br>") for cat in target_category_props["target_category"].unique()]
)
target_category_fig.update_layout(
    width=900, height=550,
    title=dict(text=f"LWS Probability across Trial Types", font=cnfg.TITLE_FONT),
    yaxis=dict(title=dict(text="P[LWS | on-target] (%)", font=cnfg.AXIS_LABEL_FONT)),
)
target_category_fig.show()

### (3) P[LWS | on-target] across *Trial Types*

In [8]:
trial_category_props = calculate_proportions(
    funnel_sizes,
    numerator_col="is_lws", denominator_col="on_target",
    primary_key="subject", subgroups="trial_category"
)

trial_category_fig = category_comparison_figure(
    trial_category_props,
    categ_col="trial_category",
    show_distributions=True,
    show_individuals=True,
    show_mean=True,
)
trial_category_fig.update_layout(
    width=900, height=550,
    title=dict(text=f"LWS Probability across Trial Types", font=cnfg.TITLE_FONT),
    yaxis=dict(title=dict(text="P[LWS | on-target] (%)", font=cnfg.AXIS_LABEL_FONT)),
)
trial_category_fig.show()

### (4) Target Rotation
We visualize the relationship between _absolute target rotation_ and the probability of LWS ($P[\text{miss}|\text{on target}]$).

In [9]:
TRIAL_SYMBOLS = {
    "all": "circle", 'BW': 'x', 'COLOR': 'diamond', "NOISE": "square",
}
TARGET_SYMBOLS = {
    'OBJECT_HANDMADE': 'cross', 'OBJECT_NATURAL': 'x',
    'ANIMAL_OTHER': 'square', 'ANIMAL_FACE': 'diamond',
    'HUMAN_OTHER': 'star', 'HUMAN_FACE': 'hexagram',
}

target_angle_props = calculate_proportions(
    funnel_sizes,
    numerator_col="is_lws", denominator_col="on_target",
    primary_key="abs_target_angle", subgroups="trial_category"
)

target_angle_fig = go.Figure()
for tr_cat in target_angle_props["trial_category"].unique():
    sym = TRIAL_SYMBOLS[tr_cat]
    color = cnfg.get_discrete_color(
        target_angle_props['trial_category'].unique().tolist().index(tr_cat), loop=True
    )
    subset_df = target_angle_props[target_angle_props["trial_category"] == tr_cat]
    target_angle_fig.add_trace(
        go.Scatter(
            name=tr_cat,
            x=subset_df["abs_target_angle"], y=subset_df["mean"],
            error_y=dict(
                type="data",
                array=subset_df["sem"],
                visible=True,
            ),
            # mode="markers+lines",
            mode="markers",
            marker=dict(size=8, symbol=sym, color=color),
            line=dict(color=color, width=2),
        )
    )

max_y = max(target_angle_props["mean"] + target_angle_props["sem"]) * 1.1
target_angle_fig.update_layout(
    title=dict(
        text="LWS Probability across Target Rotation Angles<br>(averaged across subjects)",
        font=cnfg.TITLE_FONT
    ),
    xaxis=dict(title=dict(text="Absolute Target Rotation Angle (°)")),
    yaxis=dict(title=dict(text="P[LWS | on-target]"), range=[-0.05, max_y]),
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.0,
        xanchor="right", x=1.0,
    ),
)
target_angle_fig.show()

## Statistical Modeling
We evaluate the probability of a "Looked-but-failed-to-see" error, defined as $P(\text{LWS} \mid \text{on\_target})$, using Hierarchical Logistic Regression. This approach accounts for the binary nature of the outcome and the nested structure of our data (multiple trials per subject).


### Model Specification
The relationship is modeled using a Generalized Linear Mixed Model (GLMM) with the following formula:
$$\text{logit}(is\_lws) \sim 1 + \text{trial\_category} \times \text{target\_category} \times \mid\text{target\_angle}\mid + (1 \mid \text{subject})$$
- Fixed Effects: Interaction between trial category, target category, and rotation angle.
- Random Effects: Subject-level intercepts to account for individual variability.


### Research Hypotheses
We test the influence of sensory and semantic factors on the probability of LWS errors using a maximal GLMM framework. Our analysis evaluates the following primary hypotheses:

1.  **Semantic Category Effects:**
    - **Animacy Advantage:** Animate targets (human/animal) will exhibit a lower LWS probability compared to inanimate targets (natural/handmade).
    - **Human Face Prioritization:** Within the animate category, human faces will show the lowest LWS probability, consistent with specialized neural processing for facial stimuli.
    - **Object Domain:** We will explore potential differences in LWS rates between natural and man-made inanimate objects.
2.  **Main Effect of Trial Type:** LWS probability varies significantly across image formats (*color*, *BW*, *noise*), reflecting the impact of low-level visual degradation on target verification.
3.  **Rotational Sensitivity:** Increasing the absolute rotation angle of the target ($0^\circ$–$20^\circ$) will positively correlate with LWS probability, indicating a cost for mental rotation during template matching.
4.  **Cross-Domain Interactions:** We test whether semantic advantages (e.g., the face prioritization effect) are robust across visual degradations or if they are modulated by trial type, suggesting an interaction between high-level category knowledge and low-level signal-to-noise ratios.


### Estimation & Validation
To ensure robustness, we estimate the model using two complementary frameworks and compare the resulting coefficients:
- Frequentist: Maximum Likelihood Estimation via the `glmer` package as a Python wrapper for R's `lme4` package.
- Bayesian: Hierarchical MCMC sampling via the `BAMBI` library in Python.

In [10]:
FORMULA = "is_lws ~ trial_category * target_category * target_angle + (1 | subject)"
MODEL_DATA = (
    funnel_results[["subject", "trial_category", "target_category", "target_angle", "on_target", "is_lws"]]
    .assign(abs_target_angle=lambda df: df["target_angle"].abs())
    .query("on_target == 1")
    .copy()
)

TARGET_CATEGORY_CONTRASTS = {
    "ANIMACY_EFFECT": [1/4, 1/4, 1/4, 1/4, -1/2, -1/2],     # Animate (human+animal) vs Inanimate
    "HUMAN_FACE_EFFECT": [-1/3, -1/3, -1/3, 1, 0, 0],       # Human Face vs (Human Other + Animal Other + Animal Face)
    "OBJECT_NATURALITY_EFFECT": [0, 0, 0, 0, 1, -1]         # Handmade vs Natural Inanimate Objects
}

print(f"Total number of on-target events: {MODEL_DATA.shape[0]}")
print(f"Overall P[LWS | on-target] = {100 * MODEL_DATA['is_lws'].mean() :.2f}%")

Total number of on-target events: 2341
Overall P[LWS | on-target] = 26.23%


### Frequentist Model

In [11]:
freq_model = glmer(
    formula=FORMULA,
    data=pl.from_pandas(MODEL_DATA),
    family="binomial",      # logistic regression in R uses `Binomial` (with parameter `n=1`)
)
freq_model.set_factors({
    "subject": MODEL_DATA["subject"].unique().sort_values().map(str, na_action="").tolist(),
    "trial_category": MODEL_DATA["trial_category"].unique().sort_values().tolist(),
    "target_category": MODEL_DATA["target_category"].unique().sort_values().tolist()
})
freq_result = freq_model.fit(summary=True, exponentiate=False)
print(freq_model.convergence_status)

Convergence status
: [1] TRUE
attr(,"gradient")
[1] 0.0003478183



#### Type III ANOVA Table

We conduct a **Type III Wald ANOVA test** to evaluate the significance of each model term while controlling for all other terms in the model.
Each test assesses the null hypothesis that **all coefficients associated with a given predictor or interaction are equal to zero**, conditional on the remaining predictors.

In this model, the tests evaluate whether `trial_category`, `target_category`, `target_angle`, and their interactions contribute to explaining variation in the probability of `is_lws == 1`, **within the mixed-effects model that includes a random intercept for subject**.

In [12]:
freq_model.anova()
freq_model.summary_anova()

GT(_tbl_data=shape: (7, 7)
┌─────────────────────────────────┬──────┬─────┬─────────┬────────┬──────────┬───────┐
│ model term                      ┆ df1  ┆ df2 ┆ F_ratio ┆ Chisq  ┆ p_value  ┆ stars │
│ ---                             ┆ ---  ┆ --- ┆ ---     ┆ ---    ┆ ---      ┆ ---   │
│ str                             ┆ f64  ┆ f64 ┆ f64     ┆ f64    ┆ str      ┆ str   │
╞═════════════════════════════════╪══════╪═════╪═════════╪════════╪══════════╪═══════╡
│ trial_category                  ┆ 2.0  ┆ inf ┆ 6.875   ┆ 13.75  ┆ 0.001033 ┆ **    │
│ target_category                 ┆ 5.0  ┆ inf ┆ 8.769   ┆ 43.845 ┆ <.001    ┆ ***   │
│ target_angle                    ┆ 1.0  ┆ inf ┆ 0.097   ┆ 0.097  ┆ 0.7549   ┆       │
│ trial_category:target_category  ┆ 10.0 ┆ inf ┆ 1.365   ┆ 13.65  ┆ 0.1898   ┆       │
│ trial_category:target_angle     ┆ 2.0  ┆ inf ┆ 0.88    ┆ 1.76   ┆ 0.4147   ┆       │
│ target_category:target_angle    ┆ 5.0  ┆ inf ┆ 0.393   ┆ 1.965  ┆ 0.8539   ┆       │
│ trial_category:target_category… ┆ 10.0 ┆ inf ┆ 1.232   ┆ 12.32  ┆ 0.264    ┆       │
└─────────────────────────────────┴──────┴─────┴─────────┴────────┴──────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x0000027DA25D2D20>, _boxhead=Boxhead([ColInfo(var='model term', type=<ColInfoTypeEnum.default: 1>, column_label='model term', column_align='left', column_width=None), ColInfo(var='df1', type=<ColInfoTypeEnum.default: 1>, column_label='df1', column_align='right', column_width=None), ColInfo(var='df2', type=<ColInfoTypeEnum.default: 1>, column_label='df2', column_align='right', column_width=None), ColInfo(var='F_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='F_ratio', column_align='right', column_width=None), ColInfo(var='Chisq', type=<ColInfoTypeEnum.default: 1>, column_label='Chisq', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p_value', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000027DA25D0EC0>, _spanners=Spanners([]), _heading=Heading(title='ANOVA (Type III tests)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000027DA2D8DC10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000027DA2D8E870>, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000027DA2D8DCA0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000027DA2D8EB70>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'),

#### Hypothesis #1: Semantic Category Effects
We use `emmeans()` to calculate the estimated contrasts between target categories for specific planned contrasts:
- **Animacy Effect:** Animate (human/animal) vs Inanimate (natural/handmade)
- **Human Face Effect:** Human Face vs (Human Other + Animal Other + Animal Face)
- **Object Naturalness Effect:** Handmade vs Natural Inanimate Objects
<br>
Because these are planned contrasts which are not exhaustive (i.e. they do not cover all possible pairwise comparisons), we can use the `Holm` or `Sidak` corrections for multiple comparisons, which are more powerful than the regular `Bonferroni` correction. In this case our tests are not independent (e.g. the human face effect is not independent from the animacy effect, as it is a sub-contrast of it), so we choose to use the `Holm` method because `Sidak` assumes independence between tests.

In [13]:
target_cat_marginals = (
    freq_model
    .emmeans(
        "target_category",
        contrasts={
            "ANIMATE": [0.25, 0.25, 0.25, 0.25, 0, 0],
            "INANIMATE": [0, 0, 0, 0, 0.5, 0.5],
            "HUMAN_FACE": [0, 0, 0, 1, 0, 0],
            "OTHER_ANIMATE": [1 / 3, 1 / 3, 1 / 3, 0, 0, 0],
            "OBJECT_HANDMADE": [0, 0, 0, 0, 1, 0],
            "OBJECT_NATURAL": [0, 0, 0, 0, 0, 1],
        },
        type="response",
        p_adjust="none",
    )
    .to_pandas()
    .drop(columns=["p_value", "z_ratio"])  # meaningless without contrasts
    .assign(
        prob=lambda df: df["estimate"].map(expit),
        prob_sem_approx=lambda df: df["SE"] * df["prob"] * (1 - df["prob"])  # delta method approximation
    )
)

target_cat_contrasts = freq_model.emmeans(
    "target_category",
    contrasts={
        "ANIMATE_minus_INANIMATE": [0.25, 0.25, 0.25, 0.25, -0.5, -0.5],
        "HUMAN_FACE_minus_OTHER_ANIMATE": [-1 / 3, -1 / 3, -1 / 3, 1, 0, 0],
        "OBJECT_HANDMADE_minus_OBJECT_NATURAL": [0, 0, 0, 0, 1, -1],
    },
    p_adjust="holm",
    type="response",
).to_pandas()

target_cat_contrasts

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
0,ANIMATE_minus_INANIMATE,1.954447,0.228851,inf,1.476671,2.586808,1.0,5.722883,3.141936e-08
1,HUMAN_FACE_minus_OTHER_ANIMATE,0.644026,0.083315,inf,0.472502,0.877816,1.0,-3.401347,1.341091e-03
2,OBJECT_HANDMADE_minus_OBJECT_NATURAL,0.860773,0.174745,inf,0.529445,1.399445,1.0,-0.738511,4.602039e-01


In [14]:
target_category_plot_df = (
    target_cat_marginals[["contrast", "prob", "prob_sem_approx"]]
    .rename(columns={"contrast": "contrast_group"})
    .assign(
        x_pos=lambda df: df["contrast_group"].map({
            "ANIMATE": 0,
            "INANIMATE": 1,
            "HUMAN_FACE": 3,
            "OTHER_ANIMATE": 4,
            "OBJECT_HANDMADE": 6,
            "OBJECT_NATURAL": 7,
        }),
        contrast=lambda df: df["contrast_group"].replace({
            "ANIMATE": "ANIMATE_minus_INANIMATE",
            "INANIMATE": "ANIMATE_minus_INANIMATE",
            "HUMAN_FACE": "HUMAN_FACE_minus_OTHER_ANIMATE",
            "OTHER_ANIMATE": "HUMAN_FACE_minus_OTHER_ANIMATE",
            "OBJECT_HANDMADE": "OBJECT_HANDMADE_minus_OBJECT_NATURAL",
            "OBJECT_NATURAL": "OBJECT_HANDMADE_minus_OBJECT_NATURAL",
        }),
        p_value=lambda df: df["contrast"].map(
            lambda c: target_cat_contrasts.loc[target_cat_contrasts["contrast"] == c, "p_value"].values[0]
        ),
        color=lambda df: df["contrast"].map({
            "ANIMATE_minus_INANIMATE": cnfg.get_discrete_color(0),
            "HUMAN_FACE_minus_OTHER_ANIMATE": cnfg.get_discrete_color(1),
            "OBJECT_HANDMADE_minus_OBJECT_NATURAL": cnfg.get_discrete_color(2),
        }),
        pattern=lambda df: ["" if i % 2 == 0 else "/" for i in range(len(df))]
    )
    .assign(
        contrast=lambda df: df['contrast'].map(
            lambda c: c.replace("_minus_", " - ").upper().replace("_", " ")
        ),
        contrast_group=lambda df: df['contrast_group'].map(lambda cg: cg.replace("_", " ").upper()),
    )
)

# plot the probabilities across contrasts
target_category_contrasts_fig = px.bar(
    target_category_plot_df,
    x="x_pos", y="prob", error_y="prob_sem_approx",
    color="contrast", pattern_shape="pattern",
    hover_name="contrast_group",
    hover_data={"prob": ":.4f", "contrast": True, "pattern": False, "x_pos": False},
)

# add annotations for significant contrasts
y_line = max(round(target_category_plot_df["prob"] + 1.5 * target_category_plot_df["prob_sem_approx"], 3))
for i, cntrs in enumerate(target_category_plot_df["contrast"].unique()):
    subset = target_category_plot_df[target_category_plot_df["contrast"] == cntrs]
    x_min, x_max = subset["x_pos"].min(), subset["x_pos"].max()
    target_category_contrasts_fig.add_shape(
        type="line",
        x0=x_min, x1=x_max,
        y0=y_line, y1=y_line,
        line=dict(color="black", width=1),
    )

    # add p-value annotation
    pval = subset["p_value"].iloc[0]
    if pval < 0.001:
        annotation_text = "p < 0.001"
    elif pval <= 0.05:
        annotation_text = f"p={pval:.3f}"
    else:
        annotation_text= "n.s."
    annotation_text = "<i>" + annotation_text + "</i>"
    target_category_contrasts_fig.add_annotation(
        x=(x_min + x_max) / 2, xanchor="center",
        y=y_line * 1.0, yanchor="bottom",
        text=annotation_text,
        font=cnfg.AXIS_TICK_FONT,
        showarrow=False,
    )

target_category_contrasts_fig.update_xaxes(
    title=dict(text="Target Category", font=cnfg.AXIS_LABEL_FONT),
    ticktext=target_category_plot_df["contrast_group"].map(lambda cg: cg.replace(" ", "<br>")),
    tickvals=target_category_plot_df["x_pos"],
    tickfont=cnfg.AXIS_TICK_FONT,
)
target_category_contrasts_fig.update_yaxes(
    title=dict(text="P[LWS | on-target]", font=cnfg.AXIS_LABEL_FONT),
    tickfont=cnfg.AXIS_TICK_FONT,
)
target_category_contrasts_fig.update_layout(
    width=1000, height=500,
    title=dict(
        text="Estimated LWS Probability<br><sup>Target Category Contrasts</sup>",
        font=cnfg.TITLE_FONT,
    ),
    showlegend=False,
)
target_category_contrasts_fig.show()

#### Hypothesis #1.5: Semantic Category Effects within Noise Trials
We repeat the same contrasts as in Hypothesis #1, but this time we calculate the estimated marginal means and contrasts **within the subset of trials with noisy backgrounds**.

In [15]:
target_cat_marginals_within_noise = (
    freq_model
    .emmeans(
        "target_category",
        contrasts={
            "ANIMATE": [0.25, 0.25, 0.25, 0.25, 0, 0],
            "INANIMATE": [0, 0, 0, 0, 0.5, 0.5],
            "HUMAN_FACE": [0, 0, 0, 1, 0, 0],
            "OTHER_ANIMATE": [1 / 3, 1 / 3, 1 / 3, 0, 0, 0],
            "OBJECT_HANDMADE": [0, 0, 0, 0, 1, 0],
            "OBJECT_NATURAL": [0, 0, 0, 0, 0, 1],
        },
        type="response",
        p_adjust="none",
        at={"trial_category": "NOISE",},
        apply_transforms=False,
    )
    .to_pandas()
    .drop(columns=["p_value", "z_ratio"])  # meaningless without contrasts
    .assign(
        prob=lambda df: df["estimate"].map(expit),
        prob_sem_approx=lambda df: df["SE"] * df["prob"] * (1 - df["prob"])  # delta method approximation
    )
)

target_cat_contrasts_within_noise = freq_model.emmeans(
    "target_category",
    contrasts={
        "ANIMATE_minus_INANIMATE": [0.25, 0.25, 0.25, 0.25, -0.5, -0.5],
        "HUMAN_FACE_minus_OTHER_ANIMATE": [-1 / 3, -1 / 3, -1 / 3, 1, 0, 0],
        "OBJECT_HANDMADE_minus_OBJECT_NATURAL": [0, 0, 0, 0, 1, -1],
    },
    p_adjust="holm",
    type="response",
    at={"trial_category": "NOISE",},
    apply_transforms=False,
).to_pandas()

target_cat_contrasts_within_noise

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
0,ANIMATE_minus_INANIMATE,2.318918,0.455744,inf,1.448610,3.712095,1.0,4.279692,0.000056
1,HUMAN_FACE_minus_OTHER_ANIMATE,0.789325,0.178924,inf,0.458753,1.358105,1.0,-1.043659,0.593286
2,OBJECT_HANDMADE_minus_OBJECT_NATURAL,0.790541,0.268301,inf,0.350804,1.781494,1.0,-0.692533,0.593286


In [16]:
target_category_within_noise_plot_df = (
    target_cat_marginals_within_noise[["contrast", "prob", "prob_sem_approx"]]
    .rename(columns={"contrast": "contrast_group"})
    .assign(
        x_pos=lambda df: df["contrast_group"].map({
            "ANIMATE": 0,
            "INANIMATE": 1,
            "HUMAN_FACE": 3,
            "OTHER_ANIMATE": 4,
            "OBJECT_HANDMADE": 6,
            "OBJECT_NATURAL": 7,
        }),
        contrast=lambda df: df["contrast_group"].replace({
            "ANIMATE": "ANIMATE_minus_INANIMATE",
            "INANIMATE": "ANIMATE_minus_INANIMATE",
            "HUMAN_FACE": "HUMAN_FACE_minus_OTHER_ANIMATE",
            "OTHER_ANIMATE": "HUMAN_FACE_minus_OTHER_ANIMATE",
            "OBJECT_HANDMADE": "OBJECT_HANDMADE_minus_OBJECT_NATURAL",
            "OBJECT_NATURAL": "OBJECT_HANDMADE_minus_OBJECT_NATURAL",
        }),
        p_value=lambda df: df["contrast"].map(
            lambda c: target_cat_contrasts_within_noise.loc[target_cat_contrasts_within_noise["contrast"] == c, "p_value"].values[0]
        ),
        color=lambda df: df["contrast"].map({
            "ANIMATE_minus_INANIMATE": cnfg.get_discrete_color(0),
            "HUMAN_FACE_minus_OTHER_ANIMATE": cnfg.get_discrete_color(1),
            "OBJECT_HANDMADE_minus_OBJECT_NATURAL": cnfg.get_discrete_color(2),
        }),
        pattern=lambda df: ["" if i % 2 == 0 else "/" for i in range(len(df))]
    )
    .assign(
        contrast=lambda df: df['contrast'].map(
            lambda c: c.replace("_minus_", " - ").upper().replace("_", " ")
        ),
        contrast_group=lambda df: df['contrast_group'].map(lambda cg: cg.replace("_", " ").upper()),
    )
)

# plot the probabilities across contrasts
target_category_within_noise_contrasts_fig = px.bar(
    target_category_within_noise_plot_df,
    x="x_pos", y="prob", error_y="prob_sem_approx",
    color="contrast", pattern_shape="pattern",
    hover_name="contrast_group",
    hover_data={"prob": ":.4f", "contrast": True, "pattern": False, "x_pos": False},
)

# add annotations for significant contrasts
y_line = max(round(
    target_category_within_noise_plot_df["prob"] + 1.5 * target_category_within_noise_plot_df["prob_sem_approx"], 3
))
for i, cntrs in enumerate(target_category_within_noise_plot_df["contrast"].unique()):
    subset = target_category_within_noise_plot_df[target_category_within_noise_plot_df["contrast"] == cntrs]
    x_min, x_max = subset["x_pos"].min(), subset["x_pos"].max()
    target_category_within_noise_contrasts_fig.add_shape(
        type="line",
        x0=x_min, x1=x_max,
        y0=y_line, y1=y_line,
        line=dict(color="black", width=1),
    )

    # add p-value annotation
    pval = subset["p_value"].iloc[0]
    if pval < 0.001:
        annotation_text = "p < 0.001"
    elif pval <= 0.05:
        annotation_text = f"p={pval:.3f}"
    else:
        annotation_text= "n.s."
    annotation_text = "<i>" + annotation_text + "</i>"
    target_category_within_noise_contrasts_fig.add_annotation(
        x=(x_min + x_max) / 2, xanchor="center",
        y=y_line * 1.0, yanchor="bottom",
        text=annotation_text,
        font=cnfg.AXIS_TICK_FONT,
        showarrow=False,
    )

target_category_within_noise_contrasts_fig.update_xaxes(
    title=dict(text="Target Category", font=cnfg.AXIS_LABEL_FONT),
    ticktext=target_category_plot_df["contrast_group"].map(lambda cg: cg.replace(" ", "<br>")),
    tickvals=target_category_plot_df["x_pos"],
    tickfont=cnfg.AXIS_TICK_FONT,
)
target_category_within_noise_contrasts_fig.update_yaxes(
    title=dict(text="P[LWS | on-target]", font=cnfg.AXIS_LABEL_FONT),
    tickfont=cnfg.AXIS_TICK_FONT,
)
target_category_within_noise_contrasts_fig.update_layout(
    width=1000, height=500,
    title=dict(
        text="Estimated LWS Probability<br><sup>Target Category Contrasts :: NOISE Trials</sup>",
        font=cnfg.TITLE_FONT,
    ),
    showlegend=False,
)
target_category_within_noise_contrasts_fig.show()

#### Hypothesis #2: Main Effect of Trial Type
We use `emmeans()` to calculate the estimated marginal means (predicted probabilities) for each level of `trial_category`.
<br><br>
We then calculate the pairwise contrasts between each pair of `trial_category` levels (e.g. `color` vs `BW`). Because these are post-hoc contrasts which are exhaustive (i.e. they cover all possible pairwise comparisons), we can use the `Tukey` (Tukey-HSD) correction for multiple comparisons to test which pairwise differences are statistically significant.

In [17]:
trial_cat_marginals = freq_model.emmeans(
    "trial_category", type="response", p_adjust="none"
).to_pandas()
trial_cat_contrasts = freq_model.emmeans(
    "trial_category", contrasts="pairwise", type="response", p_adjust="tukey"
).to_pandas()

trial_cat_contrasts

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions

R messages: 
NOTE: Results may be misleading due to involvement in interactions



,contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
0,COLOR / BW,0.642759,0.082543,inf,0.475703,0.868483,1.0,-3.441724,0.001675
1,COLOR / NOISE,0.910418,0.116931,inf,0.673768,1.230187,1.0,-0.730723,0.745235
2,BW / NOISE,1.416421,0.172994,inf,1.063840,1.885854,1.0,2.850396,0.012148


In [18]:
trial_category_contrasts_fig = go.Figure()
trial_category_contrasts_fig.add_trace(go.Bar(
    x=trial_cat_marginals["trial_category"],
    y=trial_cat_marginals["prob"],
    error_y=dict(
        type="data",
        array=trial_cat_marginals["SE"],
        visible=True,
    ),
    marker=dict(color=[cnfg.get_discrete_color(i, loop=True) for i in range(trial_cat_marginals.shape[0])]),
))

# add pairwise contrast annotations
for i, row in trial_cat_contrasts.iterrows():
    p_val = row["p_value"]
    if p_val > 0.05:
        continue
    group1, group2 = row["contrast"].split(" / ")
    group1_subset = trial_cat_marginals[trial_cat_marginals["trial_category"] == group1]
    x1 = group1_subset.index.values[0]
    y1 = group1_subset["prob"].values[0] + group1_subset["SE"].values[0]
    group2_subset = trial_cat_marginals[trial_cat_marginals["trial_category"] == group2]
    x2 = group2_subset.index.values[0]
    y2 = group2_subset["prob"].values[0] + group2_subset["SE"].values[0]

    # add p-value annotation
    y = max([y1, y2]) * 1.05
    trial_category_contrasts_fig.add_annotation(
        x=(x1 + x2) / 2, xanchor="center",
        y=y, yanchor="bottom",
        text=f"p={p_val:.3f}",
        font=cnfg.AXIS_TICK_FONT,
        showarrow=False,
    )

    # add line between groups
    trial_category_contrasts_fig.add_shape(
        type="line",
        x0=x1+0.25, x1=x2-0.25,
        y0=y, y1=y,
        line=dict(color="black", width=1),
    )

# update layout
trial_category_contrasts_fig.update_xaxes(
    title=dict(text="Trial Category", font=cnfg.AXIS_LABEL_FONT),
    tickfont=cnfg.AXIS_TICK_FONT,
)
trial_category_contrasts_fig.update_yaxes(
    title=dict(text="P[LWS | on target]", font=cnfg.AXIS_LABEL_FONT),
    tickfont=cnfg.AXIS_TICK_FONT,
)
trial_category_contrasts_fig.update_layout(
    width=800, height=500,
    title=dict(text=f"Estimated LWS Probability across Trial Types", font=cnfg.TITLE_FONT),
)
trial_category_contrasts_fig.show()

#### Hypothesis #3: Contrast Target Rotation (0° < 20°)
contrasting a continuous predictor is not straightforward, as the contrast is not between two discrete levels, but rather between the predicted values at two different points in the predictor space (e.g. at `target_angle=0` vs `target_angle=20`).
<br><br>
We saw above the main effect of `target_angle` is not significant, so we do not expect a significant contrast between `0°` and `20°`, but we can still calculate the predicted probabilities at these two angles, and compare them.
<br><br>
First we calculae the predicted values at `target_angle=0` and `target_angle=20` for each combination of `trial_category` and `target_category`; and then we average across all levels to get _an approximation of the marginals_ for `target_angle=0` and `target_angle=20`, and calculate the difference.

In [19]:
predicted_probs_for_target_angles = (   # for each `trial_category` ang `target_category` level
    freq_model
    .empredict(
        at={
            "target_angle": [0, 20],
            "trial_category": MODEL_DATA["trial_category"].unique().sort_values().tolist(),
            "target_category": MODEL_DATA["target_category"].unique().sort_values().tolist(),
        },
        type="response",
    )
    .to_pandas()
    .pivot(
        index=["trial_category", "target_category"],
        columns="target_angle",
        values="prob",
    )
)

# average across all levels:
predicted_probs_for_target_angles.agg(["mean", "sem"]).T.reset_index()

,target_angle,mean,sem
0,0,0.263688,0.018986
1,20,0.275066,0.027799


In [20]:
# TODO: repeat the same analyses using the Bayesian model, and compare results

### Hierarchical Bayesian Model
We use the [BAMBI package](https://bambinos.github.io/bambi/) to fit a hierarchical Bayesian logistic regression model (_logit HBM_) to the data, predicting the probability that an event that has passed the `INITIAL_STEP` will also pass all subsequent steps in the funnel (specifically, the `"final"` step), as a function of `trial category` and `target category`.

#### (1) Simple Model
In the simpler model, we only include random intercepts for each subject to account for individual differences.<br>
Our model formula is:
$$ logit(final) \sim 1 + C(trial\_category) * C(target\_category) + (1 | subject) $$

In [21]:
start = time.time()

simple_formula = "final ~ trial_category * target_category + (1 | subject)"
simple_model = bmb.Model(simple_formula, SUBSET, family="bernoulli")
simple_idata = simple_model.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)
simple_elapsed = time.time() - start
print(f"Model fitting completed in {int(simple_elapsed // 3600)}:{int((simple_elapsed % 3600) // 60)}:{simple_elapsed % 60:.2f} (hh:mm:ss)")

NameError: name 'SUBSET' is not defined

##### Model Results

In [ ]:
az.summary(
    simple_idata,
    var_names=["Intercept", "trial_category", "target_category", "trial_category:target_category"]
)

##### Model Diagnostics

In [ ]:
axes = az.plot_trace(simple_idata, var_names=["Intercept", "trial_category", "target_category"])

fig = axes.ravel()[0].figure
fig.show()

##### Model Posterior Visualization

In [ ]:
axes = az.plot_posterior(simple_idata, var_names=["Intercept", "trial_category", "target_category"],)

fig = axes.ravel()[0].figure
fig.show()

##### Posterior Contrasts Scores Distribution
For each level in the predictors (`trial_category` and `target_category`) we calculate the posterior success probability ($p[final==1|\hat\theta_{factor}]$), averaged over all subjects and all levels of the other factor.

In [ ]:
predictors = ["subject", "trial_category", "target_category"]
new_data = pd.DataFrame(product(*[SUBSET[pred].unique() for pred in predictors]), columns=predictors)
posterior_predictions_idata = simple_model.predict(idata=simple_idata, data=new_data, kind="response", inplace=False)
posterior_predictions = posterior_predictions_idata["posterior_predictive"]["final"]    # (n_cores, n_draws, n_predictors), e.g. (4, 1000, 216)

<b>Trial Category</b>

In [ ]:
column = "trial_category"
sorted_categories = pd.Series(new_data[column].unique()).astype(SUBSET[column].dtype).sort_values()
trial_category_marginals = dict()
for cat in sorted_categories:
    idx = (new_data[column] == cat).values
    predictions = np.array(posterior_predictions[:, :, idx]).flatten()
    mean, std = predictions.mean(), predictions.std()
    sem = std / np.sqrt(len(predictions))
    trial_category_marginals[cat] = (mean, std, predictions)
    print(f"{column.replace("_", " ").capitalize()}:\t{cat}\tp[final]={100*mean:.1f}±{100*std:.1f}%")

<b>Trarget Category</b>

In [ ]:
column = "target_category"
sorted_categories = pd.Series(new_data[column].unique()).astype(SUBSET[column].dtype).sort_values()
target_category_marginals = dict()
for cat in sorted_categories:
    idx = (new_data["target_category"] == cat).values
    predictions = np.array(posterior_predictions[:, :, idx]).flatten()
    mean, std = predictions.mean(), predictions.std()
    sem = std / np.sqrt(len(predictions))
    target_category_marginals[cat] = (mean, std, predictions)
    print(f"Target Category:\t{cat}\tp[final]={100*mean:.1f}±{100*std:.1f}%")

##### Posterior Contrasts
for each pair of levels in the predictors (`trial_category` and `target_category`) we compute the posterior contrast (difference) in `log-odds` scale. We then plot the distribution over each contrast of interest.<br>
NOTE: the posterior object implicitly stores contrasts against the baseline level for each factor (`COLOR` for `trial_category`; `ANIMAL_OTHER` for `target_category`), so these do not need to be computed.

In [ ]:
post = simple_idata.posterior
marginals = dict()
for factor in ["trial_category", "target_category"]:
    marginals[factor] = dict()
    base_level = SUBSET[factor].unique().sort_values()[0]   # all other levels are contrasted against this
    for (i, dim) in enumerate(post[factor][f"{factor}_dim"]):
        level = str(np.array(dim))
        if level == base_level:
            raise ValueError(f"base level {base_level} matches contrasted level {level} for factor {factor}.")
        name = f"{level}_minus_{base_level}"
        log_odds = post[factor].values[:, :, i]     # (n_draws, n_chains)
        marginals[factor][name] = log_odds

<b>Trial Category</b>

In [ ]:
marginals['trial_category']['BW_minus_NOISE'] = marginals['trial_category']['BW_minus_COLOR'] - marginals['trial_category']['NOISE_minus_COLOR']

interesting = sorted(marginals['trial_category'].keys())
_, axes = plt.subplots(1, len(interesting), figsize=(12, 3))
for i, contrast in enumerate(interesting):
    label=contrast.replace("_minus_", " - ").upper()
    az.plot_posterior(
        marginals['trial_category'][contrast], hdi_prob=0.95, color=f"C{i}", ax=axes[i],
        # label=label,
    )
    axes[i].set_title(label)
plt.show()

<b>Target Category</b>

In [ ]:
marginals['target_category']['HUMAN_minus_ANIMAL'] = 0.5 * (
        marginals['target_category']['HUMAN_OTHER_minus_ANIMAL_OTHER'] + marginals['target_category']['HUMAN_FACE_minus_ANIMAL_OTHER'] - marginals['target_category']['ANIMAL_FACE_minus_ANIMAL_OTHER']
)

marginals['target_category']['FACE_minus_OTHER'] = 0.5 * (
        marginals['target_category']['ANIMAL_FACE_minus_ANIMAL_OTHER'] + marginals['target_category']['HUMAN_FACE_minus_ANIMAL_OTHER'] - marginals['target_category']['HUMAN_OTHER_minus_ANIMAL_OTHER']
)

marginals['target_category']['ANIMATE_minus_INANIMATE'] = 0.25 * (
    marginals['target_category']['HUMAN_OTHER_minus_ANIMAL_OTHER'] + marginals['target_category']['ANIMAL_FACE_minus_ANIMAL_OTHER'] +
    marginals['target_category']['HUMAN_FACE_minus_ANIMAL_OTHER']
) - 0.5 * (
    marginals['target_category']['OBJECT_HANDMADE_minus_ANIMAL_OTHER'] + marginals['target_category']['OBJECT_NATURAL_minus_ANIMAL_OTHER']
)

interesting = ['HUMAN_minus_ANIMAL', 'FACE_minus_OTHER', 'ANIMATE_minus_INANIMATE']
_, axes = plt.subplots(1, len(interesting), figsize=(12, 3))
for i, contrast in enumerate(interesting):
    label=contrast.replace("_minus_", " - ").upper()
    data = marginals['target_category'][contrast]
    az.plot_posterior(
        data, hdi_prob=0.95, color=f"C{i}", ax=axes[i], # label=label,
    )
    axes[i].set_title(label)
plt.show()

In [ ]:
animate_probs = expit(0.25 * (
    marginals['target_category']['HUMAN_OTHER_minus_ANIMAL_OTHER'] + marginals['target_category']['ANIMAL_FACE_minus_ANIMAL_OTHER'] +
    marginals['target_category']['HUMAN_FACE_minus_ANIMAL_OTHER']
))
inanimate_probs = expit(0.5 * (
    marginals['target_category']['OBJECT_HANDMADE_minus_ANIMAL_OTHER'] + marginals['target_category']['OBJECT_NATURAL_minus_ANIMAL_OTHER']
))
prob_diff = (animate_probs - inanimate_probs).reshape(-1)
hdi_95 = 100 * az.hdi(prob_diff, hdi_prob=0.95)
print(f"Animate Targets have {100 * prob_diff.mean() :.1f} % points more chance of error than Inanimate Targets (95% HDI: [{hdi_95[0] :.1f}, {hdi_95[1] :.1f}])].")

#### (2) Complex Model
In the more complex model, we include random intercepts **and** random slopes for each subject. This will result in a better fit for subject data, by may take much longer to run. We **do not** include interaction-slopes as random (per-subject) effects, as this will result in very small sample-sizes per sub-group, and may run indefinitely.<br>
Our model formula is:
$$ logit(final) \sim 1 + C(trial\_category) * C(target\_category) + (1 + C(trial\_category) + C(target\_category) | subject) $$

In [ ]:
start = time.time()

complex_formula = "final ~ trial_category * target_category + (1 + trial_category + target_category | subject)"
complex_model = bmb.Model(complex_formula, SUBSET, family="bernoulli")
complex_idata = complex_model.fit(
    draws=2000, tune=1000, chains=4, cores=2, target_accept=0.95, progressbar=False, random_seed=SEED,
)
complex_elapsed = time.time() - start
print(f"Model fitting completed in {int(complex_elapsed // 3600)}:{int((complex_elapsed % 3600) // 60)}:{complex_elapsed % 60:.2f} (hh:mm:ss)")

##### Model Results

In [ ]:
az.summary(
    complex_idata,
    var_names=["Intercept", "trial_category", "target_category", "trial_category:target_category"]
)

##### Model Diagnostics

In [ ]:
axes = az.plot_trace(complex_idata, var_names=["Intercept", "trial_category", "target_category"])

fig = axes.ravel()[0].figure
fig.show()

##### Model Posterior Visualization

In [ ]:
axes = az.plot_posterior(complex_idata, var_names=["Intercept", "trial_category", "target_category"],)

fig = axes.ravel()[0].figure
fig.show()

### Frequentist Analysis
We repeat the Bayesian analysis using a frequentist approach - applying a generalized hierarchical linear model using the _simple_ formula:
$$logit(final) \sim 1 + C(trial\_category) * C(target\_category) + (1 | subject) $$

In [ ]:
import polars as pl
from pymer4.models import glmer

In [ ]:
model = glmer(
    formula="final ~ trial_category * target_category + (1 | subject)",
    data=pl.from_pandas(SUBSET),
    family="binomial",      # logistic regression in R uses `Binomial` (with parameter `n=1`)
)
model.set_factors({
    "subject": SUBSET["subject"].unique().sort_values().map(str, na_action="").tolist(),
    "trial_category": SUBSET["trial_category"].unique().sort_values().tolist(),
    "target_category": SUBSET["target_category"].unique().sort_values().tolist()
})
result = model.fit(summary=True, exponentiate=False)
result

##### Marginals & Contrasts
**Trial Category**

In [ ]:
model.emmeans("trial_category")

In [ ]:
model.emmeans("trial_category", contrasts="pairwise", p_adjust="tukey")

**Target Category**

In [ ]:
model.emmeans("target_category")

In [ ]:
model.emmeans(
    "target_category",
    contrasts={
        "HUMAN_minus_ANIMAL": [-0.5, 0.5, -0.5, 0.5, 0, 0],
        "FACE_minus_OTHER": [-0.5, -0.5, 0.5, 0.5, 0, 0],
        "ANIMATE_minus_INANIMATE": [0.25, 0.25, 0.25, 0.25, -0.5, -0.5]
    },
    p_adjust="sidak",
    type="response",
)